# 01 · GenAI basics

**Workshop:** AI for Actuaries — From Foundations to AI Agents
**Session / Part:** S2.P1 (reasoner)  ·  **Slides:** S1.P2.18–19
**Author:** Dr Rohan Yashraj Gupta (FIA, FIAI), with Satya Sai Mudigonda and Kasyap
**Date:** 24 July 2026 · Four Points by Sheraton, Whitefield, Bangalore
**Model:** `gemini-3.1-flash-lite` (pinned)  ·  **License:** CC BY-NC 4.0

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rohanyashraj/ifoa-workshop/blob/main/notebooks/01_genai_basics.ipynb)

## What this notebook does
Your first calls to the reasoner: a pinned Gemini call, the CCCE prompting discipline, structured JSON output, and a live look at hallucination.

*All data is hypothetical — ABC Insurer is a fictional entity for teaching only.
The story: Priya Nair (pricing, ABC General) must explain the price of policy
**ABC-MOT-047231** — a 7-year-old SUV, Tier-2, 35% NCB — so her chief actuary
**Arjun Mehta** can sign it.*

## 1. Install & authenticate
Store your key in Colab Secrets (🔑) as `GOOGLE_API_KEY`.

In [ ]:
%pip install -q google-genai

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    assert "GOOGLE_API_KEY" in os.environ, "Set GOOGLE_API_KEY (Colab Secrets or env)."

from google import genai
client = genai.Client()          # reads GOOGLE_API_KEY
MODEL = "gemini-3.1-flash-lite"  # PINNED — never 'latest'
print("Reasoner ready:", MODEL)

## 2. Your first call
The exact call an agent makes under the hood.

In [ ]:
resp = client.models.generate_content(
    model=MODEL,
    contents="Define IBNR for a board member in one sentence.",
)
print(resp.text)

## 3. CCCE — vague vs disciplined
Same request, two specifications.

In [ ]:
vague = "write about IBNR for our board"

ccce = """Task: write a 2-sentence note for our board on IBNR.
Context: motor book, FY2024, reserve strengthened by Rs 42 crore.
Constraints: use ONLY the figure above, invent no numbers, under 80 words.
Example tone: 'Reserves rose Rs X because ...'."""

for label, prompt in [("BEFORE (vague)", vague), ("AFTER (CCCE)", ccce)]:
    print("=" * 8, label, "=" * 8)
    print(client.models.generate_content(model=MODEL, contents=prompt).text, "\n")

## 4. Structured output — agents speak JSON
Guaranteed-parseable output for tools.

In [ ]:
from pydantic import BaseModel

class Factor(BaseModel):
    name: str
    relativity: float
    direction: str

resp = client.models.generate_content(
    model=MODEL,
    contents="Extract the NCB factor for policy ABC-MOT-047231 (35% NCB, relativity 0.82).",
    config={"response_mime_type": "application/json", "response_schema": Factor},
)
print(resp.text)

## 5. Hallucination, on cue
Ask for a factor that does not exist. Watch the confident fabrication — this is the *air-filter discount* the guardrail kills in notebook 04.

In [ ]:
bad = ("What is the exact 'air-filter discount' relativity in the IRDAI motor "
       "tariff for a 7-year-old SUV? Give a number.")
print(client.models.generate_content(model=MODEL, contents=bad).text)
print("\n>>> There is no such factor. The model invented one — plausibly, with a number.")

## Wrap-up
You can now: call a pinned model, prompt with CCCE, force JSON, and spot a hallucination.

**Next:** `02_models_as_tools.ipynb` — the models an agent calls.

*Demonstrated: the reasoner is powerful and confidently wrong — it needs tools that know.*